In [3]:
!pip install category_encoders

In [1]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import *
from sklearn.preprocessing import *
from sklearn.metrics import *
from lightgbm import LGBMClassifier
import gc
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from tqdm import tqdm, tqdm_notebook

import warnings
warnings.simplefilter('ignore')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import datetime
import time

import category_encoders as ce

In [3]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [4]:
prod_cols = ['P5DA', 'RIBP', '8NN1','7POT', '66FJ', 'GYSR', 'SOP4', 'RVSZ', 'PYUQ', 'LJR9', 'N2MW', 'AHXO',
             'BSTQ', 'FM3X', 'K6QO', 'QBOL', 'JWFN', 'JZ9D', 'J9JW', 'GHYX', 'ECY3']

In [5]:
miss_prod_lst = []
for p in prod_cols:
    tmp_df = train[train[p] == 1]
    tmp_df['target'] = p
    tmp_df[p] = 0  
    miss_prod_lst.append(tmp_df)
train = pd.concat(miss_prod_lst, axis=0)

In [6]:
train = train.sample(frac=1, random_state=2)
mapper = pd.Series(index = train['target'].unique(), data = np.arange(train['target'].nunique())).to_dict()
rev_mapper = {val: key for key, val in mapper.items()}
train['target'] = train['target'].map(mapper)

In [7]:
df = pd.concat([train, test]).reset_index(drop = True)
D_COL = 'join_date'
df[D_COL] = pd.to_datetime(df[D_COL], format = '%d/%m/%Y')
for f in ['month', 'year', 'week']:
    df[f] = getattr(df[D_COL].dt, f)

cat_cols = ['marital_status', 'branch_code', 'occupation_code', 'occupation_category_code', 'sex']
df[cat_cols] = df[cat_cols].apply(lambda x: pd.factorize(x)[0])
df['total_products_recommended'] = df[prod_cols].sum(axis = 1) # + 1
df['day_since_start'] = (df[D_COL] - df[D_COL].min()).dt.days

prod_cols_freq_sorted = df['target'].map(rev_mapper).value_counts().index.tolist()
prod_df = df[prod_cols].copy().astype('str')

In [8]:
for p in tqdm_notebook(prod_cols_freq_sorted):
    df[f'branch_code_{p}_sum'] = df['branch_code'].map(df.groupby('branch_code')[p].sum())

In [9]:
df['branch_occupation_nunique'] = df['branch_code'].map(df.groupby('branch_code')['occupation_code'].nunique())
df['occupation_branch_nunique'] = df['occupation_code'].map(df.groupby('occupation_code')['branch_code'].nunique())

In [10]:
from itertools import combinations


cc = list(combinations(prod_cols,2))
tmp = pd.concat([df[c[1]].sub(df[c[0]]) for c in cc], axis=1, keys=cc)
tmp.columns = tmp.columns.map(''.join)
print (tmp.shape)
df = pd.concat([df, tmp], axis=1)

(76353, 210)


In [11]:
df.loc[~df['marital_status'].isin([0,1,2]), 'marital_status'] = np.nan


In [12]:
df['prod_comb'] = ''
for p in tqdm_notebook(prod_cols_freq_sorted):
    df['prod_comb'] = df['prod_comb'] + df[p].astype('str')
df['prod_comb'] = pd.factorize(df['prod_comb'])[0]

In [13]:
df['prod_comb_freq'] = df['prod_comb'].map(df['prod_comb'].value_counts(normalize=True))

In [14]:
df[['prod_comb', 'join_date']] = df[['prod_comb', 'join_date']].apply(lambda x: pd.factorize(x)[0])

In [15]:
vc = df['prod_comb'].value_counts()
less_freq_prod_comb = vc[vc < 5].index.tolist()
df.loc[df['prod_comb'].isin(less_freq_prod_comb), 'prod_comb'] = np.nan
train, test = df[:train.shape[0]].reset_index(drop = True), df[train.shape[0]:].reset_index(drop = True)
train.shape, test.shape

((66353, 270), (10000, 270))

In [16]:
train = train[train['total_products_recommended'] <= 7]

In [17]:
cat_cols = cat_cols + ['prod_comb', 'join_date']

In [18]:
ID_COL, TARGET_COL = 'ID', 'target'
features = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]
trn, val = train_test_split(train, test_size = 0.2, stratify=train[TARGET_COL], random_state=2)

target = train[TARGET_COL]
len(features)

268

In [19]:
train_master = train.copy()
test_master = test.copy()

In [20]:

max_iter = 5
encoders =[ce.HelmertEncoder(cols = cat_cols), ce.TargetEncoder(cols = cat_cols), ce.MEstimateEncoder(cols = cat_cols), ce.BinaryEncoder(cols = cat_cols)]
oofs_all = np.zeros((len(encoders), len(train), 21))
preds_all = np.zeros((len(encoders), len(test), 21))
for i, enc in enumerate(encoders):

  print('\n-------------- Starting New Encoding -------------------------\n')
  train = train_master.copy()
  test = test_master.copy()
  train = enc.fit_transform(train, target)
  test = enc.transform(test)

  training_start_time = time.time()
  folds = GroupKFold(n_splits = max_iter)
  oofs = np.zeros((len(train), 21))
  preds_test = np.zeros((len(test), 21))

  feature_importance_df = pd.DataFrame()
  features = [c for c in train.columns if c not in [ID_COL, TARGET_COL, 'join_date']]

  for fold_, (trn_idx, val_idx) in enumerate(folds.split(train, target.values, train[ID_COL])):

      # print(f'\n---- Fold {fold_} -----\n')
      
      fold_start_time = time.time()
      
      X_trn, y_trn = train.iloc[trn_idx][features], target.iloc[trn_idx]
      X_val, y_val = train.iloc[val_idx][features], target.iloc[val_idx]
      X_test = test[features]

      X_trn = X_trn.fillna(-999)
      X_val = X_val.fillna(-999)
      X_test = X_test.fillna(-999)

      
      clf = LGBMClassifier( n_estimators = 1000000, learning_rate = 0.05, num_leaves=50, max_depth=12, colsample_bytree = 0.5, reg_alpha=2, reg_lambda=3)
      _ = clf.fit(X_trn, y_trn, eval_set = [(X_trn, y_trn), (X_val, y_val)], verbose = 100, early_stopping_rounds = 100)
      
      oofs[val_idx] = clf.predict_proba(X_val)
      current_test_pred = clf.predict_proba(X_test)
      # print(current_test_pred.shape)
      print(f'Fold {fold_} val score: {log_loss(y_val, oofs[val_idx], labels = np.arange(21))}')
      preds_test += clf.predict_proba(X_test)/max_iter

      fold_importance_df = pd.DataFrame({'feature': X_trn.columns.tolist(), 'importance': clf.feature_importances_})
      feature_importance_df = pd.concat([feature_importance_df, fold_importance_df], axis=0)
      
      fold_end_time = time.time()
      total_fold_time = int(fold_end_time - fold_start_time)
      
      # print(f"\n->-> Fold ran for {(total_fold_time)//60} minutes {(total_fold_time)%60} seconds")
      

  print(f'OOF val score: {log_loss(target, oofs)}')
  training_end_time = time.time()
  total_training_time = int(training_end_time - training_start_time)

  # print(f'\n->-> Total training time: {(total_training_time)//60} minutes {(total_training_time)%60} seconds')
  oofs_all[i] = oofs
  preds_all[i] = preds_test


-------------- Starting New Encoding -------------------------

Training until validation scores don't improve for 100 rounds.
[100]	training's multi_logloss: 0.319782	valid_1's multi_logloss: 0.404807
[200]	training's multi_logloss: 0.253258	valid_1's multi_logloss: 0.384121
[300]	training's multi_logloss: 0.223476	valid_1's multi_logloss: 0.386064
Early stopping, best iteration is:
[213]	training's multi_logloss: 0.248434	valid_1's multi_logloss: 0.383875
Fold 0 val score: 0.3831590335384676
Training until validation scores don't improve for 100 rounds.
[100]	training's multi_logloss: 0.324531	valid_1's multi_logloss: 0.381587
[200]	training's multi_logloss: 0.256989	valid_1's multi_logloss: 0.359882
[300]	training's multi_logloss: 0.226976	valid_1's multi_logloss: 0.36063
Early stopping, best iteration is:
[226]	training's multi_logloss: 0.247522	valid_1's multi_logloss: 0.359385
Fold 1 val score: 0.3593849919075305
Training until validation scores don't improve for 100 rounds.
[10

In [22]:
blnd_oofs = oofs_all.mean(axis=0)
print(f'OOF val score: {log_loss(target, blnd_oofs)}')
blnd_preds = preds_all.mean(axis=0) 

OOF val score: 0.3713339434575962


In [23]:
oofs_df = pd.DataFrame(blnd_oofs).rename(rev_mapper, axis=1)
oofs_df['ID'] = train['ID'].values
oofs_df_melted = pd.melt(oofs_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [24]:
train_cpy = train[['ID', 'target']]
train_cpy['target'] = train_cpy['target'].map(rev_mapper)
train_cpy = pd.get_dummies(train_cpy, columns = ['target'], prefix='', prefix_sep='')
train_cpy = train_cpy[oofs_df.columns.tolist()]
train_cpy_melted = pd.melt(train_cpy, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')
oofs_df_melted.loc[train_cpy_melted['target'] == 1, 'target'] = 1
log_loss(train_cpy_melted['target'], oofs_df_melted['target'])

0.011267656879214504

In [25]:
test_preds_df = pd.DataFrame(blnd_preds).rename(rev_mapper, axis=1)
test_preds_df['ID'] = test['ID'].values
test_preds_melted = pd.melt(test_preds_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [26]:
test_preds_melted['ID X PCODE'] = test_preds_melted['ID'].astype('str') + ' X ' + test_preds_melted['prod']
mapper = pd.Series(index = test_preds_melted['ID X PCODE'], data = test_preds_melted['target'].values)

In [27]:
fltr = ss['Label'] != 1
ss.loc[fltr, 'Label'] = ss.loc[fltr, 'ID X PCODE'].map(mapper)
ss['Label'].describe().round(4)
ss.to_csv('enc_all_lgb_v1.csv', index=False)

count    210000.0000
mean          0.1088
std           0.2991
min           0.0000
25%           0.0000
50%           0.0001
75%           0.0019
max           1.0000
Name: Label, dtype: float64